In [1]:
import os
from pathlib import Path
import pandas as pd
import sys
import re

In [2]:
# 1. Importamos módulo
sys.path.append("../src")
from parser import procesar_documento_hibrido
from nlp import TextEmotionAnalyzer

/home/ema/Documentos/master/repo_general/machine_learning/recu/udef-zapatero-ML/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Rutas

In [3]:
BASE_DIR = Path.cwd().parent 
RAW_PDF_PATH = BASE_DIR / "data" / "raw" / "UDEF-Zapatero-1.pdf"
PROCESSED_CSV_PATH = BASE_DIR / "data" / "processed" / "chats_completos.csv"

In [4]:
if not PROCESSED_CSV_PATH.exists():
    # Ya no pasamos la API key, el módulo parser.py se encarga de leer el .env
    df_chats = procesar_documento_hibrido(
        pdf_path=str(RAW_PDF_PATH), 
        pag_inicio=30, 
        pag_fin=154
    )
    
    # Guardamos el dataset dorado
    df_chats.to_csv(PROCESSED_CSV_PATH, index=False)
    print("\nDataset completado y guardado.")
else:
    print("El dataset ya existe, cargando desde CSV...")
    df_chats = pd.read_csv(PROCESSED_CSV_PATH)

El dataset ya existe, cargando desde CSV...


In [5]:
# Análisis final
print(f"\nTotal de mensajes extraídos: {len(df_chats)}")
print(f"Desglose de extracción por tecnología:\n{df_chats['Formato'].value_counts()}")
display(df_chats.head())


Total de mensajes extraídos: 830
Desglose de extracción por tecnología:
Formato
RegEx_F2    533
RegEx_F1    236
LLM_Groq     61
Name: count, dtype: int64


,Pagina_PDF,Formato,Fecha,Hora,Emisor,Mensaje
0,30,RegEx_F1,23/3/20,23:06:48,Miguel Palomero,Que vais a hacer con la línea aérea?
1,30,RegEx_F1,23/3/20,23:07:01,Miguel Palomero,"Has visto BA, AE y Iberia?"
2,30,RegEx_F1,23/3/20,23:07:04,Rodolfo Reyes,Aguantando la paliza
3,30,RegEx_F1,23/3/20,23:07:16,Miguel Palomero,Me imagino
4,30,RegEx_F1,23/3/20,23:07:41,Rodolfo Reyes,Necesitamos llegar a las ayudas


In [6]:
def limpieza_quirurgica(texto: str) -> str:
    """Elimina permanentemente todo el texto policial y etiquetas forenses."""
    if not isinstance(texto, str): return ""
        
    # 1. Limpieza de Audios
    texto = re.sub(r'(?i)Audio\s+[a-f0-9\-]+\.(?:opus(?:_Converted)?|mp4|wav|m4a)\s*', '', texto)
    texto = re.sub(r'(?i)Audio\s+[a-f0-9\-]+\s*', '', texto) 
    
    # 2. Amputación de las notas al pie policiales que se cuelan al final
    texto = re.sub(r'(?:\(\.\.\.\)\s*)?\d+\s*Conversaciones entre.*', '', texto, flags=re.IGNORECASE|re.DOTALL)
    texto = re.sub(r'(?:\(\.\.\.\)\s*)?\d+\s*Extraído del terminal.*', '', texto, flags=re.IGNORECASE|re.DOTALL)
    texto = re.sub(r'(?:\(\.\.\.\)\s*)?\d+\s*En el informe.*', '', texto, flags=re.IGNORECASE|re.DOTALL)
    
    # 3. Corte por marcadores policiales
    marcadores_corte = [
        r'\bEn fecha\b', r'\bEn la misma fecha\b', r'\bEl mismo día\b',
        r'\bUn día después\b', r'\bEn este sentido\b', r'\bExtraído del terminal\b',
        r'\bDatos extraídos\b', r'\bEn conexión con\b', r'\bPosteriormente\b'
    ]
    patron_corte = '|'.join(marcadores_corte)
    partes = re.split(patron_corte, texto, maxsplit=1, flags=re.IGNORECASE)
    
    return partes[0].replace('"', '').strip()

In [7]:
# Ejecución de la purga
df = pd.read_csv(PROCESSED_CSV_PATH)
filas_originales = len(df)

df['Mensaje'] = df['Mensaje'].apply(limpieza_quirurgica)
# Eliminamos si algún mensaje se quedó vacío tras limpiar o mensajes numéricos que pueden ser citas
df = df[df['Mensaje'] != ""]
df = df[~df['Mensaje'].str.isnumeric()]

In [8]:
df.to_csv(PROCESSED_CSV_PATH, index=False)
print(f"Dataset purificado permanentemente. Se conservaron {len(df)} mensajes limpios.")

Dataset purificado permanentemente. Se conservaron 830 mensajes limpios.


In [9]:
# Instanciamos el modelo de Deep Learning
nlp_model = TextEmotionAnalyzer()

# 3. Enriquecemos el dataset
df_chats_enriched = nlp_model.analyze_dataframe(df)

# 4. Guardamos el nuevo Golden Dataset con las emociones calculadas
df_chats_enriched.to_csv(PROCESSED_CSV_PATH, index=False)

Cargando modelos Transformer (RoBERTa) en memoria... Esto puede tardar un poco la primera vez.


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4178.46it/s]


Modelos NLP listos.
Analizando 830 mensajes con Deep Learning...


In [11]:
print("\n--- Muestra del enriquecimiento NLP ---")
display(df_chats_enriched[['Emisor', 'Mensaje', 'NLP_Sentimiento', 'NLP_Emocion']].head(20))


--- Muestra del enriquecimiento NLP ---


,Emisor,Mensaje,NLP_Sentimiento,NLP_Emocion
0,Miguel Palomero,Que vais a hacer con la línea aérea?,NEG,others
1,Miguel Palomero,"Has visto BA, AE y Iberia?",NEU,others
2,Rodolfo Reyes,Aguantando la paliza,NEU,others
3,Miguel Palomero,Me imagino,NEU,others
4,Rodolfo Reyes,Necesitamos llegar a las ayudas,NEU,others
5,Miguel Palomero,Así es,NEU,others
6,Rodolfo Reyes,Somos pequeños. Será mas fácil,NEU,others
7,Miguel Palomero,ERTE para empezar,NEU,others
8,Rodolfo Reyes,Ya lo hicimos,POS,others
9,Miguel Palomero,Eso es bueno,POS,joy


In [15]:
df_chats_enriched[['NLP_Sentimiento','NLP_Emocion']].groupby(["NLP_Emocion"]).count()

,NLP_Sentimiento
NLP_Emocion,
anger,13
fear,2
joy,64
others,740
sadness,6
surprise,5
